# 🔬 Napari image inspection

A clean, step-by-step workflow for opening an OME-TIFF, reading its metadata, optionally setting custom display ranges, and viewing channels in napari.

**Workflow:** ⚙️ setup → 📁 input → 📦 load → 📐 metadata → 🎚️ contrast → 🖥️ viewer


## ⚙️ 1. Prepare the environment

Set the GUI-related environment variables required by napari in this session.

In [1]:
import os
os.environ["DISPLAY"] = ":1"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-jovyan"
os.environ["QT_QPA_PLATFORM"] = "xcb"

In [ ]:
import napari
from bioio import BioImage

# Add channel names and [min, max] values here to override napari's defaults.
CUSTOM_CONTRAST_LIMITS = {}


## 📁 2. Choose the input image

Point the workflow to the OME-TIFF that should be inspected. Update `file_path` when working with another dataset.

In [ ]:
from pathlib import Path

# 📁 Choose the input image
file_path = Path("your_image_file.ome.tif")
input_path = file_path.expanduser().resolve()
print(f"Input image: {input_path}")

## 📦 3. Load the image data

Read the first timepoint in `CZYX` order so the channel axis is explicit for later display.

In [ ]:
img = BioImage(input_path)
data = img.get_image_data("CZYX", T=0)

print(f"Loaded data with shape CZYX = {data.shape}")

## 📐 4. Read physical scale and channel metadata

Extract voxel spacing, channel names, and OME-defined colors so napari can display the image in physical coordinates with meaningful labels.

In [ ]:
scale = img.physical_pixel_sizes
spatial_scale = [scale.Z, scale.Y, scale.X]

ome = img.ome_metadata
channels_meta = ome.images[0].pixels.channels
channel_names = [channel.name or f"Channel {index}" for index, channel in enumerate(channels_meta)]

colormaps = []
for channel in channels_meta:
    if channel.color is not None:
        red, green, blue = channel.color.as_rgb_tuple()[:3]
        colormaps.append([[0, 0, 0], [red / 255, green / 255, blue / 255]])
    else:
        colormaps.append("gray")

print(f"Channels: {', '.join(channel_names)}")
print(f"Spatial scale (Z, Y, X): {spatial_scale}")

## 🎚️ 5. Set optional display contrast

Use `CUSTOM_CONTRAST_LIMITS` to set `[minimum, maximum]` values for selected channels. Channels without custom values keep napari's automatic contrast limits.


In [ ]:
contrast_limits = [CUSTOM_CONTRAST_LIMITS.get(name) for name in channel_names]

print("CUSTOM_CONTRAST_LIMITS = {")
for name, limits in zip(channel_names, contrast_limits):
    if limits is not None:
        print(f'    "{name}": [{limits[0]}, {limits[1]}],')
print("}")


Create one viewer with additive channel blending, OME colors, physical scaling, and any custom contrast limits.


In [ ]:
viewer = napari.Viewer()

image_kwargs = {
    "channel_axis": 0,
    "scale": spatial_scale,
    "name": channel_names,
    "colormap": colormaps,
    "blending": "additive",
}
if CUSTOM_CONTRAST_LIMITS:
    image_kwargs["contrast_limits"] = contrast_limits

viewer.add_image(data, **image_kwargs)
